In [2]:
# cella per importare le librerie necessarie

import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets

from  sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score

In [9]:
from pathlib  import Path
data_path = Path('../data/raw')

files = {'grid':'trentino-grid.geojson',
         'adm_reg':'administrative_regions_Trentino.json',
        'weather':'meteotrentino-weather-station-data.json',
        'precip':'precipitation-trentino.csv',
        'precip-avail':'precipitation-trentino-data-availability.csv',
        'SET-1':'SET-nov-2013.csv',
        'SET-2':'SET-dec-2013.csv',
        'SET-lines':'line.csv',
        'twitter':'social-pulse-trentino.geojson'}

In [10]:
# Creo il DataFrame geopandas con i dati meteo

import json
from shapely.geometry import Point
df_grid = gpd.read_file(data_path / files['grid'])

with open(data_path / files['weather']) as f:
    weather_json = json.load(f)

weather_df = gpd.GeoDataFrame(weather_json['features'])

# Introduco la colonna geometry, che contiene le posizioni delle stazioni meteo di rilevamento
weather_df['geometry'] = weather_df['geomPoint.geom'].apply(lambda x:Point(x['coordinates'][0], x['coordinates'][1]))
weather_df.drop(columns=['geomPoint.geom'],inplace=True)


# La colonna del vento è divisa nella forma forza_del_vento@direzione_azimutale. Vogliamo separare questi due dati in due diverse colonne
winds_cols = [c for c in weather_df.columns if c.startswith("winds.")]

new_cols = []

for c in winds_cols:

    split_cols = weather_df[c].str.split("@", n=1, expand=True)

    # ensure two columns always exist
    split_cols = split_cols.reindex(columns=[0, 1])

    split_cols.columns = [f"{c}_spd", f"{c}_dir"]

    split_cols = split_cols.apply(pd.to_numeric, errors="coerce")

    new_cols.append(split_cols)

weather_df = pd.concat([weather_df] + new_cols, axis=1)
weather_df.drop(columns=winds_cols,inplace=True)

/var/folders/kk/q9vd_8nn5w5cp7fcpz9zs4sh0000gn/T/ipykernel_77609/2619868955.py:13: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  weather_df['geometry'] = weather_df['geomPoint.geom'].apply(lambda x:Point(x['coordinates'][0], x['coordinates'][1]))


In [12]:
# modifica del dataframe in modo da mettere in ordine cronologico

meteo_df = None

for station_id in weather_df['station'].drop_duplicates():

    df_station = weather_df[weather_df["station"] == station_id]

    # === COLONNE PRECIPITAZIONI ===
    temp_cols = [c for c in df_station.columns if c.startswith("temperatures.")]
    prec_cols = [c for c in df_station.columns if c.startswith("precipitations.")]
    winds_cols_spd = [c for c in df_station.columns if (c.startswith("winds.") and c.endswith("_spd"))]
    winds_cols_dir = [c for c in df_station.columns if (c.startswith("winds.") and c.endswith("_dir"))]

    # lista finale
    rows = []
    prec = []
    winds_spd = []
    winds_dir = []

    # === COSTRUZIONE NUOVO DATAFRAME ===
    for _, row in df_station.iterrows():

        date = pd.to_datetime(row["date"])

        for i in range(len(temp_cols)):

            # estrae HHMM
            hhmm = temp_cols[i].split(".")[1]

            hour = hhmm[:2]
            minute = hhmm[2:]

            # costruzione datetime
            dt = date.replace(
                hour=int(hour),
                minute=int(minute)
            )

            # formato richiesto:
            # minuto-ora-giorno-mese-anno
            # esempio 1215140313
            custom_time = dt.strftime("%M%H%d%m%y")

            rows.append({
                'datetime': custom_time,
                'temperatures_' + station_id: row[temp_cols[i]],
                'precipitations_' + station_id: row[prec_cols[i]],
                'winds_spd_' + station_id: row[winds_cols_spd[i]],
                'winds_dir_' + station_id: row[winds_cols_dir[i]]
            }) 
    
    df = pd.DataFrame(rows)

    if meteo_df is None:
        meteo_df = df
    else:
        meteo_df = meteo_df.merge(df, on="datetime")